# 7-тарау. Көршілер, ағаштар және топтар

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 7.1. Ирис гүлдерінің деректері

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target      # X: (150, 4), y: 0, 1 немесе 2
print(X.shape, iris.target_names)

# 70% — оқыту, 30% — тест
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 7.2. k-жақын көршілер

In [ ]:
def knn_predict(X_train, y_train, x, k=5):
    # 1. x-тен барлық оқыту мысалдарына дейінгі қашықтықтар
    distances = np.sqrt(((X_train - x) ** 2).sum(axis=1))
    # 2. ең жақын k мысалдың индекстері
    nearest = np.argsort(distances)[:k]
    # 3. көршілердің кластары бойынша дауыс беру
    votes = np.bincount(y_train[nearest])
    return votes.argmax()

preds = np.array([knn_predict(X_train, y_train, x) for x in X_test])
accuracy = (preds == y_test).mean()
print(f"kNN дәлдігі: {accuracy:.1%}")

In [ ]:
for k in [1, 3, 5, 15, 51]:
    preds = np.array([knn_predict(X_train, y_train, x, k) for x in X_test])
    print(f"k = {k:>2}: дәлдік {(preds == y_test).mean():.1%}")

## 7.3. Шешім ағаштары

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)
print("Тест дәлдігі:", round(tree.score(X_test, y_test), 3))

names = ["тост_ұзын", "тост_ені", "күлте_ұзын", "күлте_ені"]
print(export_text(tree, feature_names=names))

In [ ]:
for name, imp in zip(names, tree.feature_importances_):
    print(f"{name:12s} {imp:.3f}")

## 7.4. k-орташа: мұғалімсіз топтау

In [ ]:
rng = np.random.default_rng(5)
centers = np.array([[2, 2], [7, 3], [4.5, 7.5]])
points = np.vstack([c + rng.normal(0, 0.9, (40, 2)) for c in centers])

def kmeans(X, k, steps=10):
    centroids = X[:k].copy()                   # бастапқы орталықтар
    for _ in range(steps):
        # әр нүктеден әр орталыққа дейінгі қашықтық: (n, k)
        d = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        labels = d.argmin(axis=1)              # ең жақын орталық
        centroids = np.array([X[labels == j].mean(axis=0) for j in range(k)])
    return centroids, labels

centroids, labels = kmeans(points, k=3)
print(centroids.round(2))
print("Топтардағы нүктелер саны:", np.bincount(labels))